In [2]:
import pandas as pd
import numpy as np

words = pd.read_csv("words.csv", keep_default_na=False)

In [3]:
print(words.dtypes)

Word            str
Char Count    int64
Value         int64
dtype: object


In [4]:
words.head()

,Word,Char Count,Value
0,aa,2,2
1,aah,3,10
2,aahed,5,19
3,aahing,6,40
4,aahs,4,29


In [5]:
# pd.to_numeric converts text to numbers.
# errors="coerce" turns anything that cannot be converted (here x) into NaN instead of crashing. 
# This is the standard way to clean a messy numeric column.

In [6]:
s = pd.Series(["10", "20", "x", "40"])
print(pd.to_numeric(s, errors="coerce"))

0    10.0
1    20.0
2     NaN
3    40.0
dtype: float64


In [7]:
# astype("int16") stores numbers in 2 bytes instead of 8. It is safe here because the values are below 32,767.
# int8 holds up to 127, which fits the letter counts (max 28).

In [8]:
small = words.copy()
small["Value"] = small["Value"].astype("int16")
small["Char Count"] = small["Char Count"].astype("int8")

In [9]:
print(words.memory_usage(deep=True).sum())

12804167


In [10]:
print(small.memory_usage(deep=True).sum())

10557468


In [11]:
d = pd.DataFrame({"name": [" Amit", "Riya ", "Amit", None], "score": [85, np.nan, 85, 70]})
print(d)

    name  score
0   Amit   85.0
1  Riya     NaN
2   Amit   85.0
3    NaN   70.0


In [12]:
print(d.isna().sum())   # isna().sum() counts the missing values per column. There is one in each.

name     1
score    1
dtype: int64


In [13]:
print(d.fillna({"name": "unknown", "score": d["score"].mean()}))
# fillna with a dictionary fills each column with its own value. 
# Text gets “unknown” and the score gets the column mean (80.0).

      name  score
0     Amit   85.0
1    Riya    80.0
2     Amit   85.0
3  unknown   70.0


In [14]:
print(d.dropna())   # dropna() removes any row that has a missing value, so only rows 0 and 2 remain.

    name  score
0   Amit   85.0
2   Amit   85.0


In [15]:
print(d.dropna(subset=["score"]))  
# dropna(subset=["score"]) looks only at the score column, so row 3 (missing name only) stays.

    name  score
0   Amit   85.0
2   Amit   85.0
3    NaN   70.0


In [16]:
print(d.assign(name=d["name"].str.strip()))
# str.strip() removes spaces at both ends. Before it, 
# " Amit" and "Amit" looked different. After it, rows 0 and 2 are the same.

   name  score
0  Amit   85.0
1  Riya    NaN
2  Amit   85.0
3   NaN   70.0


In [17]:
print(d.assign(name=d["name"].str.strip()).drop_duplicates(subset=["name", "score"]))
# drop_duplicates(subset=[...]) keeps the first copy of each repeated combination. Row 2 was removed.

   name  score
0  Amit   85.0
1  Riya    NaN
3   NaN   70.0


In [18]:
print(d.drop_duplicates(subset=["name"]))

    name  score
0   Amit   85.0
1  Riya     NaN
2   Amit   85.0
3    NaN   70.0


In [19]:
clean = words.rename(columns={"Char Count": "char_count", "Value": "value", "Word": "word"})
print(clean.columns.tolist())
print(clean.drop(columns=["value"]).columns.tolist())
print(words.columns.str.lower().str.replace(" ", "_").tolist())
# columns.str.lower().str.replace(" ", "_") cleans every column name at once. 
# This is a standard first step on any file.

['word', 'char_count', 'value']
['word', 'char_count']
['word', 'char_count', 'value']


In [20]:
clean["avg_letter"] = (clean["value"] / clean["char_count"]).round(2)  # round(2) keeps 2 decimals
print(clean.head())

     word  char_count  value  avg_letter
0      aa           2      2        1.00
1     aah           3     10        3.33
2   aahed           5     19        3.80
3  aahing           6     40        6.67
4    aahs           4     29        7.25


In [21]:
clean["size"] = np.where(clean["char_count"] >= 10, "long", "short")
# np.where(condition, value_if_true, value_if_false)
print(clean["size"].value_counts())

size
short    105241
long      67582
Name: count, dtype: int64


In [22]:
print(clean["size"].head())

0    short
1    short
2    short
3    short
4    short
Name: size, dtype: str


In [23]:
clean["band"] = pd.cut(clean["char_count"], bins=[0, 5, 9, 14, 30], labels=["tiny", "short", "medium", "long"])
print(clean["band"].value_counts().sort_index())

band
tiny      13607
short     91634
medium    60118
long       7464
Name: count, dtype: int64


In [24]:
# pd.cut sorts numbers into ranges. 
# Each range includes its right end: (0, 5], (5, 9], (9, 14], (14, 30]. 
# So 5 letters is “tiny”, 9 is “short” and 14 is “medium”. 
# The new column has dtype category.

In [25]:
clean["band2"] = np.select([clean["char_count"] <= 6, clean["char_count"] <= 10], ["tiny", "short"], default="large")
print(clean["band2"].value_counts())

band2
short    96704
large    47280
tiny     28839
Name: count, dtype: int64


In [26]:
# initial command
# clean = words.rename(columns={"Char Count": "char_count", "Value": "value", "Word": "word"})

In [27]:
print(clean["char_count"].size)

172823


In [28]:
len(clean["char_count"])          # Total number of rows

172823

In [29]:
clean["char_count"].count()       # Number of non-missing values

np.int64(172823)

In [30]:
clean["char_count"].isna().sum()  # Number of missing values

np.int64(0)

## Sorting

In [31]:
print(clean.sort_values("value", ascending=True).head()[["word", "char_count", "value"]])

      word  char_count  value
0       aa           2      2
19      ab           2      3
10456   ba           2      3
10457  baa           3      4
20     aba           3      4


In [32]:
print(clean.sort_values("value", ascending=False).head()[["word", "char_count", "value"]])

                            word  char_count  value
126623   reinstitutionalizations          23    319
91369   microspectrophotometries          24    317
91370     microspectrophotometry          22    309
91367    microspectrophotometers          23    308
103503    otorhinolaryngologists          22    307


In [33]:
print(clean.sort_values("char_count", ascending=True).head(4)[["word", "char_count", "value"]])

       word  char_count  value
0        aa           2      2
47631    em           2     18
71976    id           2     13
103248   os           2     34


In [34]:
print(clean.sort_values("char_count", ascending=False).head(4)[["word", "char_count", "value"]])

                               word  char_count  value
51072  ethylenediaminetetraacetates          28    287
51071   ethylenediaminetetraacetate          27    268
47241   electroencephalographically          27    269
72762     immunoelectrophoretically          25    307


In [35]:
x = words.copy()

In [36]:
print(x.head())

     Word  Char Count  Value
0      aa           2      2
1     aah           3     10
2   aahed           5     19
3  aahing           6     40
4    aahs           4     29


In [37]:
# str.upper() changes case. str.len() gives the length. Similar: str.lower(), str.title().

In [38]:
print(x["Word"].str.len().head(2).tolist())

[2, 3]


In [39]:
print(x["Word"].str.upper().head(2).tolist())

['AA', 'AAH']


## Counting and searching

In [40]:
x["vowels"] = x["Word"].str.count("[aeiou]")
print(x.head(10))

       Word  Char Count  Value  vowels
0        aa           2      2       2
1       aah           3     10       2
2     aahed           5     19       3
3    aahing           6     40       3
4      aahs           4     29       2
5       aal           3     14       2
6     aalii           5     32       4
7    aaliis           6     51       4
8      aals           4     33       2
9  aardvark           8     76       3


In [41]:
print(x["Word"].str.contains("ing").sum())
print(x["Word"].str.contains("ing"))

15241
0         False
1         False
2         False
3          True
4         False
          ...  
172818    False
172819    False
172820    False
172821    False
172822    False
Name: Word, Length: 172823, dtype: bool


In [42]:
print(x["Word"].str.endswith("ing").sum())
print(x["Word"].str.endswith("ing"))

12564
0         False
1         False
2         False
3          True
4         False
          ...  
172818    False
172819    False
172820    False
172821    False
172822    False
Name: Word, Length: 172823, dtype: bool


In [43]:
print(x["Word"].str.startswith("un").sum())
print(x["Word"].str.startswith("un").head())

3856
0    False
1    False
2    False
3    False
4    False
Name: Word, dtype: bool


In [44]:
print(x["Word"].str.replace("a", "@", regex=False).head(5).tolist()) 
# str.replace(old, new) swaps text. regex=False treats old as plain text.

['@@', '@@h', '@@hed', '@@hing', '@@hs']


## Spliting and Cleaning names

In [45]:
t = pd.Series(["Amit Shah", " Riya Rao "])
print(t.str.strip().str.title().tolist())
# str.strip().str.title() chains methods: trim spaces, then capitalise each word.

['Amit Shah', 'Riya Rao']


In [46]:
print(t.str.strip().str.split(" ", expand=True))   # str.split(" ", expand=True) splits text into separate columns.

      0     1
0  Amit  Shah
1  Riya   Rao


In [47]:
print(t.str.split().str[0].tolist())

['Amit', 'Riya']


In [48]:
print(t.str.split().str[1].tolist())

['Shah', 'Rao']


In [49]:
# Read it left to right: group by Char Count, take the Value column, take the mean of each group. 

In [50]:
print(words.groupby("Char Count")["Value"].mean().round(2).head())
# So 2-letter words average a value of 22.58.

Char Count
2    22.58
3    35.17
4    47.49
5    60.29
6    70.13
Name: Value, dtype: float64


In [51]:
print(words[["Word","Char Count","Value"]].head())

     Word  Char Count  Value
0      aa           2      2
1     aah           3     10
2   aahed           5     19
3  aahing           6     40
4    aahs           4     29


In [52]:
print(words.groupby("Char Count")["Value"].mean().head())

Char Count
2    22.583333
3    35.167695
4    47.488342
5    60.289602
6    70.126247
Name: Value, dtype: float64


In [53]:
print(words.groupby("Char Count")["Value"].agg([ "mean", "count","min" ,"max"]).head(4))

                 mean  count  min  max
Char Count                            
2           22.583333     96    2   45
3           35.167695    972    4   66
4           47.488342   3903    6   88
5           60.289602   8636    8  111


In [54]:
print(words.groupby("Char Count").agg(total=("Value", "sum"), avg=("Value", "mean")).round(1).head(3))

             total   avg
Char Count              
2             2168  22.6
3            34183  35.2
4           185347  47.5


### merge: join tables on a shared column
### This is the pandas version of an SQL JOIN.

In [65]:
lookup = pd.DataFrame({"band": ["tiny", "short", "medium", "long"], "label": ["T", "S", "M", "L"], "fee": [1, 2, 3, 4]})
m = words.merge(lookup, on="band", how="left")
print(m[["Word", "band", "label", "fee"]].head(3))

    Word  band label  fee
0     aa  tiny     T    1
1    aah  tiny     T    1
2  aahed  tiny     T    1


In [66]:
print(m["fee"].isna().sum(), len(m), len(words))

0 172823 172823


In [72]:
# how=	Keeps
# inner	only ids found in both tables
# left	all rows of the left table, matches from the right
# right	all rows of the right table
# outer	all rows from both

In [67]:
a = pd.DataFrame({"id": [1, 2, 3], "v": ["a", "b", "c"]})
b = pd.DataFrame({"id": [2, 3, 4], "w": ["x", "y", "z"]})

In [68]:
print(a.merge(b, on="id", how="inner"))

   id  v  w
0   2  b  x
1   3  c  y


In [70]:
print(a.merge(b, on="id", how="left"))

   id  v    w
0   1  a  NaN
1   2  b    x
2   3  c    y


In [71]:
print(a.merge(b, on="id", how="outer"))

   id    v    w
0   1    a  NaN
1   2    b    x
2   3    c    y
3   4  NaN    z


In [73]:
print(a.merge(b, on="id", how="outer", indicator=True)["_merge"].tolist())

['left_only', 'both', 'both', 'right_only']


In [76]:
words.head(3)

,Word,Char Count,Value,band
0,aa,2,2,tiny
1,aah,3,10,tiny
2,aahed,5,19,tiny


In [77]:
words.tail(3)

,Word,Char Count,Value,band
172820,zymurgy,7,135,short
172821,zyzzyva,7,151,short
172822,zyzzyvas,8,170,short


In [75]:
c = pd.concat([words.head(3), words.tail(3)], ignore_index=True)
print(c)

       Word  Char Count  Value   band
0        aa           2      2   tiny
1       aah           3     10   tiny
2     aahed           5     19   tiny
3   zymurgy           7    135  short
4   zyzzyva           7    151  short
5  zyzzyvas           8    170  short


In [78]:
print(a)

   id  v
0   1  a
1   2  b
2   3  c


In [79]:
print(b)

   id  w
0   2  x
1   3  y
2   4  z


In [80]:
print(pd.concat([a,b]))

   id    v    w
0   1    a  NaN
1   2    b  NaN
2   3    c  NaN
0   2  NaN    x
1   3  NaN    y
2   4  NaN    z


In [81]:
print(pd.concat([a,b],axis=1))

   id  v  id  w
0   1  a   2  x
1   2  b   3  y
2   3  c   4  z


In [86]:
print(pd.concat([a,b],axis=1,ignore_index=True))

   0  1  2  3
0  1  a  2  x
1  2  b  3  y
2  3  c  4  z


In [87]:
sales = pd.DataFrame({"date": ["2026-01-05", "2026-01-20", "2026-02-03", "2026-02-15"], "amt": [100, 150, 200, 50]})
sales["date"] = pd.to_datetime(sales["date"])
print(sales.dtypes)

date    datetime64[us]
amt              int64
dtype: object


In [88]:
print(sales)

        date  amt
0 2026-01-05  100
1 2026-01-20  150
2 2026-02-03  200
3 2026-02-15   50


In [89]:
sales["month"] = sales["date"].dt.month
sales["day_name"] = sales["date"].dt.day_name()
sales["year"] = sales["date"].dt.year
print(sales)

        date  amt  month day_name  year
0 2026-01-05  100      1   Monday  2026
1 2026-01-20  150      1  Tuesday  2026
2 2026-02-03  200      2  Tuesday  2026
3 2026-02-15   50      2   Sunday  2026


In [100]:
print(sales.groupby("month")["amt"].sum())   # Group by month for monthly totals (both months total 250).

month
1    250
2    250
Name: amt, dtype: int64


In [95]:
print(sales.groupby("month")["amt"].head())

0    100
1    150
2    200
3     50
Name: amt, dtype: int64


In [98]:
print(sales[sales["date"] >= "2026-02-01"])  # Real dates can be compared with plain date strings like "2026-02-01".

        date  amt  month day_name  year
2 2026-02-03  200      2  Tuesday  2026
3 2026-02-15   50      2   Sunday  2026


In [99]:
print((sales["date"].max() - sales["date"].min()).days)
# Subtracting dates gives a time difference, and .days extracts the day count. Here it is 41 days.

41


In [101]:
# ┌──────────────────┬──────────────────────────────────────────────────────────────────┐
# │ TASK             │ TOOLS                                                            │
# ├──────────────────┼──────────────────────────────────────────────────────────────────┤
# │ Load and save    │ read_csv, read_excel, to_csv(index=False)                        │
# ├──────────────────┼──────────────────────────────────────────────────────────────────┤
# │ First look       │ shape, head, info, describe, isna().sum(), value_counts          │
# ├──────────────────┼──────────────────────────────────────────────────────────────────┤
# │ Select           │ df["a"], df[["a","b"]], loc, iloc, df[mask], query               │
# ├──────────────────┼──────────────────────────────────────────────────────────────────┤
# │ Clean            │ astype, to_numeric(errors="coerce"), fillna, dropna,             │
# │                  │ drop_duplicates, rename, str.strip                               │
# ├──────────────────┼──────────────────────────────────────────────────────────────────┤
# │ New columns      │ arithmetic, np.where, np.select, pd.cut, map, apply, assign      │
# ├──────────────────┼──────────────────────────────────────────────────────────────────┤
# │ Text             │ .str.lower/upper/contains/replace/split/len                      │
# ├──────────────────┼──────────────────────────────────────────────────────────────────┤
# │ Summaries        │ groupby + agg (named), transform, pivot_table, crosstab          │
# ├──────────────────┼──────────────────────────────────────────────────────────────────┤
# │ Combine          │ merge (how = inner, left, outer), concat                         │
# ├──────────────────┼──────────────────────────────────────────────────────────────────┤
# │ Dates            │ to_datetime, .dt.year/month/day_name, resample, Timedelta        │
# ├──────────────────┼──────────────────────────────────────────────────────────────────┤
# │ Style            │ method chaining with assign and query                            │
# └──────────────────┴──────────────────────────────────────────────────────────────────┘